# RS 다이나믹스 — S&P500 스크리너

이 노트북은 실행 버튼만 누르면 되는 껍데기다. 실제 로직은 드라이브에 올려둔
`.py` 파일 안에 있고, 여기서는 그걸 호출하고 결과를 보기만 한다.

**처음 한 번만** — 드라이브에 `RS_APP` 폴더를 만들고 받은 파일을 전부 넣는다.

**매번** — 셀 1 → 셀 2(확인) → 셀 3(전체) 순으로 실행.


## 1. 준비

드라이브를 연결하고 폴더로 이동한 뒤 패키지를 깐다. 권한 팝업이 뜨면 허용.
마지막 출력에 `필요한 파일: 전부 있음` 이 뜨면 준비 완료.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/RS_APP'   # 폴더 이름이 다르면 여기만 고친다
%cd $FOLDER

!pip -q install yfinance lxml

import os
need = ['run_sp500.py', 'fetch_data.py', 'screener_live.py',
        'snapshot_export.py', 'app_template.html']
missing = [f for f in need if not os.path.exists(f)]
print('\n필요한 파일:', '전부 있음' if not missing else f'없음 -> {missing}')
print('폴더 내용:', sorted(os.listdir('.')))


## 2. 빠른 확인 (60종목)

전체를 돌리기 전에 60종목으로 파이프라인이 도는지 본다. 3~5분.

출력에서 이 줄을 확인할 것:

```
[스크린] 유니버스 60 · RS>=80 N · 패턴 적중 M
```

- `RS>=80` 이 10 안팎이면 정상 (상위 20%)
- 1~2개뿐이면 가격 행렬이 깨진 것 → 맨 아래 문제 해결 참고


In [ ]:
!python run_sp500.py --limit 60


## 3. 전체 실행 (500종목)

처음 한 번은 20~40분. 대부분 실적 데이터를 받는 시간이다.
같은 날 다시 돌리면 `cache/` 덕분에 훨씬 빠르다.

급하면 `--no-fundamentals` 를 붙여 실적 수집을 건너뛴다.
EPS 계단과 실적 D-day 가 비지만 스크리닝은 다 나온다.


In [ ]:
!python run_sp500.py


## 4. 결과 훑어보기

앱을 열지 않고도 오늘 뭐가 걸렸는지 여기서 바로 본다.


In [ ]:
import json, pandas as pd

snap = json.load(open('latest.json', encoding='utf-8'))
df = pd.DataFrame(snap['tickers'])
m = snap['market']

print(f"기준일 {snap['meta']['as_of']} · {snap['meta']['universe']} · {len(df)}종목")
print(f"시장: {m['m_gauge']} · 분산일 {m['distribution_days']} · "
      f"200일선 위 {m['pct_above_200ma']}%\n")

cols = [c for c in ['ticker', 'name', 'rs', 'rs_quad', 'rs_traj', 'setup',
                    'base_count', 'ud_volume', 'tt_pass', 'grp_rs', 'theme',
                    'd_to_earn', 'eps_accel', 'stop1_pct'] if c in df.columns]

hits = df[df.setup.notna()].sort_values('rs', ascending=False)
print(f'-- 패턴 적중 {len(hits)}종목 --')
display(hits[cols].reset_index(drop=True))


In [ ]:
print('-- 4분면 --')
display(df.rs_quad.value_counts().rename('종목수').to_frame())

print('-- 패턴별 --')
display(df.setup.value_counts().rename('종목수').to_frame())

print('-- 테마 강도 (RS85 이상 종목 수) --')
display(pd.DataFrame(snap['themes'])[['name', 'rs85_count', 'count']].head(12))


In [ ]:
# 주도 분면 + 그룹RS 강함 + 패턴 보유 — 실제로 볼 만한 후보만 추린다
pick = df[(df.rs_quad == '주도') & df.setup.notna() &
          (df.grp_rs.fillna(0) >= 80)].sort_values('rs', ascending=False)
print(f'{len(pick)}종목')
display(pick[cols].reset_index(drop=True))


## 5. 앱 미리보기

노트북 안에서 앱을 그대로 띄운다. 카드를 누르면 상세가 열리고 탭도 동작한다.
핀치 줌은 마우스로 안 되니 4분면 확대는 폰에서 볼 것.


In [ ]:
import html as _html
from IPython.display import HTML

src = open('app.html', encoding='utf-8').read()
HTML('<iframe srcdoc="' + _html.escape(src, quote=True) + '" '
     'style="width:400px;height:780px;border:1px solid #22314A;'
     'border-radius:14px;background:#0B111B"></iframe>')


## 6. 내려받기

`app.html` 은 데이터가 들어간 실행본이다. 받아서 더블클릭하면 열리고,
폰에 옮겨 브라우저로 열면 실제 사용감 그대로다.
드라이브에도 저장돼 있으니 폰 드라이브 앱에서 열어도 된다.


In [ ]:
from google.colab import files
files.download('app.html')
# files.download('latest.json')   # 앱에 원격으로 물릴 때 쓰는 스냅샷


## 7. 파라미터 만지기

패턴이 너무 안 걸리거나 너무 많이 걸릴 때. 값을 바꾼 뒤 3번을 다시 돌린다.
캐시가 있어서 두 번째부터는 금방 끝난다.

| 값 | 뜻 | 올리면 |
|---|---|---|
| `pivot_within` | 피벗이 전고점 대비 얼마 안쪽인가 | 더 많이 걸린다 |
| `vcp_max_last_depth` | 마지막 수축 허용 깊이 | VCP가 늘어난다 |
| `max_base_days` | 베이스 최대 길이 | 오래된 베이스도 잡힌다 |
| `accept_after_bo` | 돌파 후 며칠까지 수용 | 이미 뜬 종목이 섞인다 |


In [ ]:
import screener_live as sl
for k, v in vars(sl.PatternParams()).items():
    print(f'{k:24} {v}')


In [ ]:
# 값을 바꾸려면 파일을 직접 고친다. 아래 두 줄의 주석을 풀고 원하는 값으로.
# 고친 뒤 3번 셀을 다시 실행하면 반영된다.

OLD, NEW = 'pivot_within: float = 0.25', 'pivot_within: float = 0.35'

# s = open('screener_live.py', encoding='utf-8').read()
# assert OLD in s, '원본 문자열을 못 찾았다 — 값이 이미 바뀌었는지 확인'
# open('screener_live.py', 'w', encoding='utf-8').write(s.replace(OLD, NEW))
# print('수정 완료:', NEW)


## 문제 해결

**`RS>=80` 종목이 몇 개뿐이다**  
가격 행렬이 깨진 것이다. `[위생]` 줄에서 제거된 행이 지나치게 많은지 보고,
`!python run_sp500.py --refresh` 로 가격을 새로 받는다.

**패턴 적중이 0이다**  
7번에서 `pivot_within` 을 0.35 로 올려본다.

**야후에서 가격을 못 받는다**  
레이트 리밋이면 배치가 통째로 빈다. 시간을 두고 재시도.

**실적 수집이 너무 느리다**  
`!python run_sp500.py --no-fundamentals` 로 스크리닝만 먼저 돌린다.

**세션이 끊겼다**  
캐시가 드라이브에 있으니 셀 1부터 다시 돌리면 받은 데이터는 재사용된다.
